# Customer Support Agent in 4 Hours

Fine-tune Qwen 3.5 0.8B into a customer-support agent using StateSet Agents' GSPO trainer + multi-turn `ConversationEnvironment`. This is the framework's headline showcase — what the X/LinkedIn audience wants: a developer training a model overnight for a business use case.

**Estimated runtime:** ~3 hours on a Colab A100. **Cost:** ~$2.

## What this notebook does

1. Pins the framework to commit `14c0e65`.
2. Sets seed `42` across all RNGs.
3. Loads the bundled 24-scenario customer-support corpus across 4 intents.
4. Evaluates the un-fine-tuned baseline (composite reward: intent ack + brand voice + safety).
5. Fine-tunes with **GSPO** on multi-turn dialogues.
6. Re-evaluates + writes a schema-compliant JSON result.
7. Serves the trained agent via the bundled FastAPI service for a live demo.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/customer_support_4h.ipynb)

Set runtime to A100 for the published timings.

## 1. Pin + install

In [ ]:
import os
import subprocess
import sys

PINNED_COMMIT = '71786b9'  # whitepaper v0.12.2 + all framework fixes from GSM8K v1 session

if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
print('Pinned to', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
%pip install --quiet -e '.[training,api]'
%pip install --quiet accelerate bitsandbytes datasets
# Colab pins older versions that don't recognise newer model_types (e.g. 'qwen3_5')
# and newer peft requires torchao >= 0.16.0. Upgrade to the latest stable.
# After this finishes, restart the runtime once so the upgraded packages replace
# the versions already imported by Colab's preloaded modules.
%pip install --quiet -U transformers accelerate peft trl torchao
print('Install complete. If this is the first run, do Runtime > Restart session, then re-run from cell 1.')

## 2. Seeds + data

In [ ]:
from stateset_agents.utils.reproducibility import set_all_seeds
from stateset_agents.data import (
    load_support_scenarios,
    make_support_scenarios,
    SupportRewardComposite,
)

SEED = 42
state = set_all_seeds(SEED)
print('Seeds applied:', state.to_dict())

all_scenarios = load_support_scenarios()
train_scenarios = all_scenarios[:16]
eval_scenarios = all_scenarios[16:]

print(f'\nTotal scenarios: {len(all_scenarios)}')
print(f'Train: {len(train_scenarios)} ({set(s.intent for s in train_scenarios)})')
print(f'Eval:  {len(eval_scenarios)}  ({set(s.intent for s in eval_scenarios)})')
print('\nSample:')
for s in train_scenarios[:3]:
    print(f' [{s.intent}] {s.user_query}')

## 3. Baseline evaluation

How does the un-fine-tuned model score on our composite reward? This sets the floor we have to beat.

In [ ]:
from stateset_agents.core import MultiTurnAgent
from stateset_agents.core.agent_config import AgentConfig
from stateset_agents.core.trajectory import ConversationTurn

MODEL_NAME = 'Qwen/Qwen3.5-0.8B'

def prompt_for(s):
    return (
        'You are a helpful customer support agent. Respond to the user warmly, '
        'address their concern directly, and confirm the next step.\n\n'
        f'User: {s.user_query}\n\nAgent:'
    )

async def evaluate(agent, scenarios):
    reward = SupportRewardComposite()
    scores = []
    for s in scenarios:
        response = await agent.generate_response(prompt_for(s))
        turns = [ConversationTurn(role='assistant', content=response)]
        result = await reward.compute_reward(turns, context=s.to_scenario())
        scores.append(result.score)
    return sum(scores) / max(len(scores), 1), scores

# MultiTurnAgent, not the abstract base Agent (generate_response is abstract on Agent).
baseline_agent = MultiTurnAgent(AgentConfig(
    model_name=MODEL_NAME,
    max_new_tokens=320,
    temperature=0.0,
    do_sample=False,
    torch_dtype='bfloat16',
    attn_implementation='sdpa',  # Colab has no flash-attn; sdpa is native PyTorch.
))
await baseline_agent.initialize()  # top-level await — Jupyter already has a running loop.
baseline_score, baseline_per = await evaluate(baseline_agent, eval_scenarios)
print(f'Baseline composite score: {baseline_score:.3f}')
for s, v in zip(eval_scenarios, baseline_per):
    print(f'  [{s.intent}] {v:.2f}  — {s.user_query[:60]}…')

## 4. Fine-tune with GSPO

Multi-turn training with the bundled `ConversationEnvironment`. GSPO's sequence-level importance ratios make it stable on the multi-turn rollouts (see §5.2 of the whitepaper).

In [ ]:
from stateset_agents.training import GSPOConfig, train_with_gspo
from stateset_agents.core import ConversationEnvironment, MultiTurnAgent
import time

config = GSPOConfig(
    model_name=MODEL_NAME,
    num_generations=4,
    clip_range_left=3e-4,
    clip_range_right=4e-4,
    learning_rate=5e-6,
    max_prompt_length=512,
    max_completion_length=320,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    gradient_checkpointing=True,
    num_epochs=3,
    warmup_ratio=0.1,
    output_dir='/content/gspo_support',
)

agent = MultiTurnAgent(AgentConfig(
    model_name=MODEL_NAME,
    torch_dtype='bfloat16',
    attn_implementation='sdpa',
))
# Do NOT call agent.initialize() — train_with_gspo loads the model itself.

env = ConversationEnvironment(
    scenarios=make_support_scenarios(train_scenarios),
    reward_fn=SupportRewardComposite(),
    max_turns=4,
)

# train_with_gspo's default scenario→query mapping looks for `context`/`task`/`metadata`
# keys. SupportScenario.to_scenario() emits {user_query, intent, must_acknowledge,
# must_avoid} — none of those keys match, so the trainer would default to the
# literal 'Hello' as the prompt and drop the rubric. Pass explicit train_queries
# so the support reward gets must_acknowledge/must_avoid/intent in context.
train_queries = [
    {
        'prompt': prompt_for(s),
        'context': {
            'must_acknowledge': list(s.must_acknowledge),
            'must_avoid': list(s.must_avoid),
            'intent': s.intent,
        },
    }
    for s in train_scenarios
]

t0 = time.time()
trained_agent = await train_with_gspo(
    config=config,
    agent=agent,
    environment=env,
    reward_model=env.reward_fn,
    train_queries=train_queries,
)
train_wall_clock = time.time() - t0
print(f'\nTraining wall-clock: {train_wall_clock:.0f}s ({train_wall_clock/60:.1f}m)')

## 5. Post-training evaluation

In [ ]:
final_score, final_per = await evaluate(agent, eval_scenarios)  # top-level await
print(f'Final composite score: {final_score:.3f}')
print(f'Improvement: {final_score - baseline_score:+.3f}')
print()
for s, b, f in zip(eval_scenarios, baseline_per, final_per):
    delta = f - b
    print(f'  [{s.intent}] {b:.2f} → {f:.2f}  ({delta:+.2f})  {s.user_query[:50]}…')

## 6. Save the schema-compliant result

In [ ]:
import json
import torch
from datetime import datetime, timezone
from pathlib import Path

result = {
    'trainer': 'gspo',
    'task': 'customer_support',
    'model': MODEL_NAME,
    'seed': SEED,
    'commit': PINNED_COMMIT,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'config': {
        'num_generations': config.num_generations,
        'clip_range_left': config.clip_range_left,
        'clip_range_right': config.clip_range_right,
        'learning_rate': config.learning_rate,
        'lora_r': config.lora_r,
    },
    'metrics': {
        'eval_pass_at_1': final_score,                # composite-reward score
        'eval_pass_at_1_baseline': baseline_score,
        'improvement': final_score - baseline_score,
        'wall_clock_seconds': train_wall_clock,
        'train_examples': len(train_scenarios),
        'eval_examples': len(eval_scenarios),
        'peak_vram_mb': torch.cuda.max_memory_allocated() // (1024**2) if torch.cuda.is_available() else 0,
    },
    'hardware': {
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'cuda': torch.version.cuda if torch.cuda.is_available() else None,
    }
}

out = Path(f'/content/gspo_seed{SEED}_customer_support_qwen3_5_0_8b.json')
out.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
print(f'\nSaved to: {out}')

## 7. Live demo — serve via FastAPI

The same trained agent now serves through the bundled FastAPI service. This is the production-deployment path.

In [ ]:
# The trained agent can be served via InferenceService for production use; here we
# just call it directly to demonstrate the trained policy on out-of-distribution queries.
test_queries = [
    'I need a refund for order #9981',
    'The app keeps crashing when I open it',
    'What are your business hours?',
]
for q in test_queries:
    response = await agent.generate_response(prompt_for(
        type('Q', (), {'user_query': q})()
    ))  # top-level await
    print(f'\nQ: {q}')
    print(f'A: {response[:300]}')

## 8. Next steps

- **Run with 2 more seeds** (1337, 2026) for variance bars — required for whitepaper publication.
- **Try GRPO and DAPO** for the same config — three notebooks, three trainers.
- **Scale the dataset** — swap the bundled 24-scenario corpus for a real customer-support log (~500-2000 trajectories). The framework supports any list-of-dicts with the same schema.
- **Add an LLM-judge reward** — replace the rule-based composite with `LLMJudgeReward` for production use.
- **Deploy** — `helm install stateset-agents deployment/helm/ -f deployment/helm/values-a100.yaml` after publishing your LoRA adapter to a model registry.